In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}}\right)$$

In [5]:
#1. 位置编码✅
class positional_encoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # d_model：每个词向量的维度 max_len：句子最大长度（每个i 的位置信息是确定的，提前写好，长度更大时也可以外推）
        # 初始化位置编码矩阵为0矩阵，形状为（max_len, d_model）
        pe = torch.zeros(max_len, d_model)
        
        # 定义每个token位置的索引：0 1 2 3 4 ...... max_len - 1
        # 扩展一维：position 的形状为 [max_len, 1] 方便后续与缩放因子相乘
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) #前闭后开
        
        # div_term 每个维度得到缩放因子，torch.arange(0, d_model, 2)：生成偶数维度索引：0 2 4 ......对应公式：2i
        # (torch.arange(0, d_model, 2) * （-math.log(10000.0) / d_model） ：（2i/d_model）* （-ln（10000.0））
        # exp：e的指数，math.log: ln
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -math.log(10000.0) / d_model)
        
        # 每个 token的位置索引 pos * 每个维度的缩放因子（div_term）再套上 sin 得到偶数维度的位置编码值
        pe[:, 0::2] = torch.sin(position * div_term)#切片得到偶数位置  0 2 4 ...
        # 每个 token的位置索引 pos * 每个维度的缩放因子（div_term）再套上 cos 得到奇数维度的位置编码值
        pe[:, 1::2] = torch.cos(position * div_term)#切片得到奇数位置  1 3 5 ...
        
        # 增加batch_size维度，(1, max_len, d_model) 方便后续与输入的 token embedding 相加
        pe = pe.unsqueeze(0)
        
        # 注册为 buffer ，把位置编码 pe 存在模型里面，不参与训练，但是随着模型保存/加载
        self.register_buffer('pe', pe)

    def forward(self, x):
        # X ：输入的 embedding 形状 ：（batch_size, seq_len, d_model)
        seq_len = x.size(1)
        
        # 每个token 的embedding加上对应的位置编码
        # self.pe[:, :seq_len, :] 取前seq_len 个位置，形状变为（1, seq_len, d_model) 可以和输入 x 对齐相加
        x = x + self.pe[:, :seq_len, :] # (1, seq_len, d_model) “1” 广播
        
        return x


        
        

$\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$

In [ ]:
#2. 多头自注意力机制✅
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0 # d_model 必须能被 num_heads 整除

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads 

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    
    
    def split_heads(self, x):
        # x.shape = (bs, seq_len, d_model) → (bs, num_heads, seq_len, head_dim)
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        # 为什么一定要把 num_heads 放到前面？
        # 因为 PyTorch 在做矩阵乘法时，默认是对最后两个维度进行计算的。把 num_heads 提到了前面，
        # 相当于把不同的“头”当成了独立的批次（Batch）维度。
        # 这样，后面的注意力机制就会自动、并行地在每一个头内部计算，头与头之间互不干扰。
        x = x.transpose(1, 2) # (bs, num_heads, seq_len, head_dim)
        
        return x



    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        ######
        # Q, K, V: (batch_size, num_heads, seq_len, head_dim)
        # K.transpose(-2, -1)：把 K 的最后两个维度倒过来，变成 (bs, num_heads, head_dim, seq_len)
        # torch.matmul(Q, K^T)：
        # Q 的形状：(bs, num_heads, seq_len, head_dim)
        # K^T 的形状：(bs, num_heads, head_dim, seq_len)
        # 它们相乘（只看最后两维：[seq_len, head_dim] × [head_dim, seq_len]），
        # 得到的 scores 形状为 (bs, num_heads, seq_len, seq_len)。
        # 这个 scores 矩阵里，每一个元素代表一句话里，第 i 个单词和第 j 个单词的关联度（分数）。
        ######
        # / math.sqrt(self.head_dim)：为什么要除以分母？
        # 因为如果维度 head_dim 很大，点积算出来的值会非常大，送进 Softmax 后会导致梯度消失。除以这个缩放因子可以稳定数值。
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        #掩码机制：如果传入了 mask，把那些不需要关注的位置（比如 Padding 占位符，或者文本生成时未成熟的未来单词）
        # 强行赋值为 -10^9（接近负无穷）。这样在下一步做 Softmax 时，这些位置的权重就会变成 0，彻底被忽略。
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # dim=-1 代表对张量的最后一维进行 Softmax 计算。
        # 我们来看进入 Softmax 前，scores 的维度形状：shape = (batch_size, num_heads, seq_len, seq_len)
        # 为了看清最后一维是什么，我们固定前三个维度，只看最后两个维度。最后两维构成了一个形状为 (seq_len, seq_len) 的二维方阵：
        # 矩阵的行（Row）：代表当前正准备发出注意力的 Token（即 Q）。
        # 矩阵的列（Column）：代表被关注的 Token（即 K）。
        # 为什么要沿列（dim=-1）做 Softmax？
        # 因为自注意力机制的逻辑是：“对于当前行的某一个单词，它对整句话所有单词的注意力权重加起来应该等于 1”。
        # dim=-1（最后一维）对应的正是列。对最后一维做 Softmax，意味着把每一行里的所有列分值进行归一化。
        # 例如：一句话有 3 个词，针对第 1 个词（第 1 行），Softmax 会把这行中对应第 1、2、3 个词的分数变成概率，让它们相加等于 1。
        attn_weights = F.softmax(scores, dim=-1)
        # torch.matmul(attn_weights, V)：
        # 权重形状：(bs, num_heads, seq_len, seq_len)
        # V 的形状：(bs, num_heads, seq_len, head_dim)
        # 相乘后，得到 output 形状：(bs, num_heads, seq_len, head_dim)。
        output = torch.matmul(attn_weights, V)
        # 这就是经过注意力加权后的、每个头各自吐出的新特征向量。
        
        return output



    def merge_heads(self, x):
        # x.shape = (bs, num_heads, seq_len, head_dim) → (bs, seq_len, d_model)
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2) # (bs, seq_len, num_heads, head_dim)
        x = x.reshape(batch_size, seq_len, self.d_model)
        return x


    def forward(self, Q, K, V, mask=None):
        # Q,K,V.shape = (bs, seq_len, d_model) 
        # 1. 经过线性层映射，并立刻拆分成多头
        Q = self.split_heads(self.Wq(Q))
        K = self.split_heads(self.Wk(K))
        V = self.split_heads(self.Wv(V))

        # 2. 并行计算每个头内部的注意力
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask) # (bs, num_heads, seq_len, head_dim)

        # 3. 把多头结果合并，并过最后一层线性映射进行特征融合
        output = self.merge_heads(attn_output)
        output = self.Wo(output)
        return output


In [7]:
#3. 前馈神经网络✅ 
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear2(self.dropout((F.relu(self.linear1(x)))))
        return x


In [8]:
#4. Encoder层✅

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # 在 PyTorch 中，任何一个继承自 nn.Module 的类（比如你定义的 MultiHeadAttention），
        # 它需要接收什么参数、接收多少个参数，全部由它自己的 forward 函数决定。
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        #多头自注意力机制+残差+归一化
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        #前馈神经网络 + 残差 + 归一化
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x



In [9]:
#5. Decoder层✅
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    # x: 当前 Decoder 侧收到的输入张量（形状通常为 [Batch_Size, Tgt_Seq_Len, d_model]）。
    # encoder_output: Encoder 最终输出的特征矩阵（包含了输入源句子的全部语义信息）。
    # src_mask: 编码器掩码，主要是为了盖住源句子里的 Padding（补零符号）。
    # tgt_mask: 解码器掩码（因果掩码），防止模型在训练时“偷看未来”的词。
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        #Mask 自注意力机制
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # 通俗理解：Decoder 拿着自己当前手里的词（Query），扭头去问 Encoder 吐出来的上下文（Key）：
        # “你看我手里的词和你们哪部分最相关？”，对齐之后，从 Encoder 侧把最相关的特征（Value）给抽取过来。
        # 使用的是 src_mask：因为是在跟 Encoder 的输出打交道，所以这里需要用 src_mask 来遮盖掉源语言中无意义的 Padding 符号。
        #cross注意力 （Q来自decoder，K和V来自encoder）
        cross_output = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_output))

        #前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x



In [10]:
#6. Transformer模型✅
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                num_layers=6, d_ff=2048, max_seq_len=5000, dropout=0.1 ):
        super().__init__()

        #Embedding层
        #nn.Embedding 负责把词表里的索引转化成长度为 d_model（如 512）的特征向量。
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = positional_encoding(d_model, max_seq_len)

        #Encoder 层
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        #Decoder 层
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        #输出层
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model


    def generate_mask(self, src, tgt):
        # 源序列的 padding mask
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2) # (batch, 1, 1, src_len)
 
        # 目标序列的 padding mask
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3) # (batch, 1, tgt_len, 1)

        # 目标序列 casual mask（防止看到未来的信息）
        seq_len = tgt.size(1)
        
        # torch.tril 生成了一个下三角矩阵
        nopeak_mask = torch.tril(torch.ones(1, seq_len, seq_len)).bool().to(tgt.device)
        tgt_mask = tgt_mask & nopeak_mask

        return src_mask, tgt_mask


    def forward(self, src, tgt):
        # src:(batch, seq_len), tgt:(batch, seq_len)
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        #Encoder
        src_emb = self.dropout(self.pos_encoding(self.src_embedding(src)))
        enc_output = src_emb
        for layer in self.encoder_layers:
            enc_output = layer(enc_output, src_mask)
        
        #Decoder
        tgt_emb = self.dropout(self.pos_encoding(self.tgt_embedding(tgt)))
        dec_output = tgt_emb
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)
        
        #输出层
        output = self.fc_out(dec_output)
        
        return output
        
        


In [ ]:
#7. 测试模型✅
# 模型参数
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
dropout = 0.1

# 创建模型
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, 
                    num_layers, d_ff, dropout=dropout)

# 生成随机输入
batch_size = 2
src_seq_len = 10
tgt_seq_len = 8

# torch.randint(low, high, size) 是用来专门生成随机整数张量的函数
src = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
tgt = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))

# src：模拟了作为提问/输入的源语言数据（比如翻译任务中的中文原句）。
# tgt：模拟了作为答案/目标的目标语言数据（比如翻译任务中的英文目标句）。
# 前向传播
output = model(src, tgt)

print(f"源序列形状: {src.shape}")
print(f"目标序列形状: {tgt.shape}")
print(f"输出形状: {output.shape}")
print(f"\n模型总参数量: {sum(p.numel() for p in model.parameters()):,}")
print("\n✅ 模型运行成功！")

源序列形状: torch.Size([2, 10])
目标序列形状: torch.Size([2, 8])
输出形状: torch.Size([2, 8, 5000])

模型总参数量: 51,823,496

✅ 模型运行成功！


In [12]:
#8. 模型结构可视化
print(model)

Transformer(
  (src_embedding): Embedding(5000, 512)
  (tgt_embedding): Embedding(5000, 512)
  (pos_encoding): positional_encoding()
  (encoder_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (Wq): Linear(in_features=512, out_features=512, bias=True)
        (Wk): Linear(in_features=512, out_features=512, bias=True)
        (Wv): Linear(in_features=512, out_features=512, bias=True)
        (Wo): Linear(in_features=512, out_features=512, bias=True)
      )
      (feed_forward): FeedForward(
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x DecoderLaye